In [ ]:
"""
sandbox_bweight_decoder.ipynb

Characterize encoding profiles of neurons most informative in decoding strategy.

Author: Stellina X. Ao
Created: 2026-07-29
Last Modified: 2026-07-29
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251024_142407"  # "20251028_140930" # "20251027_152036"

## init

In [ ]:
import numpy as np
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id, norm=True)
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

In [ ]:
plt.figure()
plt.plot(encoder.trial_data["strategy"])
plt.show()

In [ ]:
np.where(encoder.trial_data["strategy"] == 1)
np.where(encoder.trial_data["strategy"] == -1)

In [ ]:
from core.data import get_strategy_filter_idxs

idxs_all = get_strategy_filter_idxs(encoder.trial_data, cond_balance=True)
idxs = np.sort(np.concatenate((idxs_all["mb"], idxs_all["mf"])))

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import KFold

X = encoder.robs[idxs]
y = encoder.trial_data["strategy"].iloc[idxs]

decoder = LogisticRegressionCV(Cs=np.logspace(-5, 5, 11, base=10)).fit(X, y)
scores_cv = np.zeros(5)

for i, (train_idx, test_idx) in enumerate(
    KFold(n_splits=5, shuffle=True, random_state=2).split(X, y)
):
    decoder = LogisticRegressionCV(Cs=np.logspace(-5, 5, 11, base=10)).fit(
        X[train_idx], y.iloc[train_idx]
    )
    scores_cv[i] = decoder.score(X[test_idx], y.iloc[test_idx])

print(scores_cv.mean())

In [ ]:
# neurons with the largest weight, what is their task variable encoding like?
plt.figure()
plt.imshow(
    encoder_mf.encoder_weights[np.argsort(decoder.coef_.ravel()), 5:],
    aspect="auto",
    vmin=-5,
    vmax=5,
    cmap="coolwarm",
)
plt.colorbar()
plt.show()